# Install mandatory libraries

In [1]:
!pip install ultralytics pycocotools --no-dependencies --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.2 MB/s eta 0:00:00


# Initialization

## Import core libraries

In [2]:
import pandas as pd
import numpy as np
import json
import yaml
from ultralytics import YOLO, settings
import torch
import random
import os

FOLD_CSV = '/kaggle/input/datasets/nttgaming/ripvis-cs431/fold_management.csv'
DATASET_PATH = '/kaggle/working/data/train'
IOU_THRESHOLDS = np.arange(0.40, 0.96, 0.05)
SEED = 42

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Constants

## Connect WanDB

## Set seed for deterministic

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [4]:
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("wandb"))
settings.update({"wandb": True})

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: minhhoanghsftg to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
dataset_yaml = f'dataset.yaml'
with open(dataset_yaml, 'w') as f:
    yaml.dump({
        'path' : '/kaggle/input/datasets/lakechamplain/ripvis-cs231/ripvis_cs231', 
        'train': 'train/images', 
        'val'  : 'val/images', 
        'test' : 'test/images', 
        'nc': 1, 'names': ['rip_currents']}, f)

# Patch CBAM to use for YAML model loading

In [6]:
# ============================================================
# PATCH: CBAM support — Kaggle DDP + Colab compatible
# No reload. Pickle-safe. Workers get CBAM on fresh import.
# ============================================================
import ultralytics
import ultralytics.nn.tasks as _tasks
from ultralytics.nn.modules.conv import CBAM
from pathlib import Path

_tasks_path = Path(_tasks.__file__)
_init_path  = Path(ultralytics.__file__).parent / "nn" / "modules" / "__init__.py"

# ── 1. modules/__init__.py — export CBAM so tasks.py import block can see it ──
_init_src = _init_path.read_text()
if "CBAM" not in _init_src:
    _init_src = _init_src.replace(
        "from .conv import (",
        "from .conv import (\n    CBAM,",
        1,
    )
    if "__all__" in _init_src:
        _init_src = _init_src.replace(
            "__all__ = (",
            '__all__ = (\n    "CBAM",',
            1,
        )
    _init_path.write_text(_init_src)

# ── 2. tasks.py — add CBAM to import block + elif in parse_model ──────────────
_src = _tasks_path.read_text()

if "    CBAM,\n" not in _src:
    _src = _src.replace(
        "    CBLinear,\n    Classify,",
        "    CBLinear,\n    CBAM,\n    Classify,",
        1,
    )

if "if m is CBAM:" not in _src:
    _src = _src.replace(
        "        if m in base_modules:\n",
        "        if m is CBAM:\n"
        "            c2 = ch[f]\n"
        "            args = [c2]\n"
        "\n"
        "        elif m in base_modules:\n",
        1,
    )

_tasks_path.write_text(_src)

# ── 3. Live globals injection — main process sees CBAM immediately ─────────────
_tasks.CBAM = CBAM

# ── 4. Re-exec parse_model — picks up new elif without module reload ───────────
#    SegmentationModel untouched → no PicklingError on torch.save
_lines = _src.splitlines()
_start = next(i for i, l in enumerate(_lines) if l.startswith("def parse_model("))
_end   = next(
    (i for i in range(_start + 1, len(_lines))
     if _lines[i] and not _lines[i][0].isspace() and _lines[i][0] not in "#"),
    len(_lines),
)
exec(compile("\n".join(_lines[_start:_end]), str(_tasks_path), "exec"), vars(_tasks))

print("✅ CBAM ready — main process + DDP workers")

✅ CBAM ready — main process + DDP workers


# Prepare model configuration

In [7]:
model_yaml_content = """ 
# Parameters
nc: 1 # number of classes
end2end: True # whether to use end-to-end mode
reg_max: 1 # DFL bins
scales: # model compound scaling constants, i.e. 'model=yolo26n-seg.yaml' will call yolo26-seg.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

# YOLO26n backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]                   # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]                  # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]                  # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]                  # 5-P4/16
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]                 # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5, 3, True]]           # 9
  - [-1, 2, C2PSA, [1024]]                      # 10

# YOLO26n head
head:  
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 11
  - [[-1, 6], 1, Concat, [1]]                   # 12 - cat backbone P4
  - [-1, 2, C3k2, [512, True]]                  # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]  # 14
  - [[-1, 4], 1, Concat, [1]]                   # 15
  - [-1, 2, C3k2, [256, True]]                  # 16 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]                  # 17
  - [[-1, 13], 1, Concat, [1]]                  # 18
  - [-1, 2, C3k2, [512, True]]                  # 19 (P4/16-medium)
  - [-1, 1, CBAM, []]                           # 20

  - [-1, 1, Conv, [512, 3, 2]]                  # 21
  - [[-1, 10], 1, Concat, [1]]                  # 22
  - [-1, 1, C3k2, [1024, True, 0.5, True]]      # 23 (P5/32-large)

  - [[16, 20, 23], 1, Segment26, [nc, 32, 256]] # 24
"""

with open("/kaggle/working/yolo26-seg-cbam-p4.yaml", "w") as f:
    f.write(model_yaml_content)

# Model training

In [8]:
model = YOLO("/kaggle/working/yolo26n-seg-cbam-p4.yaml").load("yolo26n-seg.pt")

Transferred 408/847 items from pretrained weights


In [9]:
train_args = {
    # --- Training configurations ---
    "data": dataset_yaml,
    "val": True,
    "seed": SEED,
    "workers": 2,
    "patience": 20,
    "device": [0, 1],

    # --- Saving strategy ---
    "save": True,
    "save_period": 10,

    # --- Model configurations ---
    "imgsz": 640,
    "max_det": 50,

    # --- Optimizer ---
    "epochs": 100,
    "batch": 128,
    "cos_lr": True,
    "warmup_epochs": 10,
    "optimizer": "MuSGD",
    "lr0": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,

    # --- Augmentation ---
    "hsv_h": 0.05,
    "flipud": 0.5,

    # --- Others ---
    "project": "UIT-RipVis",
    "name": "YOLO26n-seg-CBAM_P4-CS231",
}

In [10]:
model.train(**train_args)

Ultralytics 8.4.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.05, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=50, mixup=0.0, mode=train, model=/kaggle/working/yolo26n-seg-cbam-p4.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLO26n-

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: minhhoanghsftg to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/runs/segment/UIT-RipVis/YOLO26n-seg-CBAM_P4-CS231/wandb/run-20260508_000438-YOLO26n-seg-CBAM_P4-CS231_20260508_000438
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run YOLO26n-seg-CBAM_P4-CS231
wandb: ⭐️ View project at https://wandb.ai/minhhoanghsftg/UIT-RipVis
wandb: 🚀 View run at https://wandb.ai/minhhoanghsftg/UIT-RipVis/runs/YOLO26n-seg-CBAM_P4-CS231_20260508_000438


Transferred 408/847 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.6±0.4 ms, read: 55.2±28.0 MB/s, size: 376.9 KB)
train: Scanning /kaggle/input/datasets/lakechamplain/ripvis-cs231/ripvis_cs231/train/labels... 14316 images, 3648 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14316/14316 271.3it/s 52.8s
WARNING ⚠️ train: Cache directory /kaggle/input/datasets/lakechamplain/ripvis-cs231/ripvis_cs231/train is not writable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 45.4±22.4 MB/s, size: 448.7 KB)
val: Scanning /kaggle/input/datasets/lakechamplain/ripvis-cs231/ripvis_cs231/val/labels... 2725 images, 1345 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2725/2725 240.6it/s 11

wandb: uploading artifact run_YOLO26n-seg-CBAM_P4-CS231_20260508_000438_model; updating run metadata; uploading artifact run-YOLO26n-seg-CBAM_P4-CS231_20260508_000438-curvesPrecision-RecallB_table--nVzaA; uploading artifact run-YOLO26n-seg-CBAM_P4-CS231_20260508_000438-curvesF1-ConfidenceB_table-QpCemg; uploading artifact run-YOLO26n-seg-CBAM_P4-CS231_20260508_000438-curvesPrecision-ConfidenceB_table-VfXySA (+ 5 more)
wandb: uploading artifact run_YOLO26n-seg-CBAM_P4-CS231_20260508_000438_model; uploading artifact run-YOLO26n-seg-CBAM_P4-CS231_20260508_000438-curvesPrecision-RecallB_table--nVzaA; uploading artifact run-YOLO26n-seg-CBAM_P4-CS231_20260508_000438-curvesF1-ConfidenceB_table-QpCemg; uploading artifact run-YOLO26n-seg-CBAM_P4-CS231_20260508_000438-curvesPrecision-ConfidenceB_table-VfXySA; uploading artifact run-YOLO26n-seg-CBAM_P4-CS231_20260508_000438-curvesRecall-ConfidenceB_table-dJg4Dw (+ 4 more)
wandb: uploading artifact run_YOLO26n-seg-CBAM_P4-CS231_20260508_000438_mod

# Evaluate trained model

In [11]:
settings.update({"wandb": False})

In [12]:
from __future__ import annotations

import json, os, platform, sys, tempfile, warnings, io
from contextlib import redirect_stdout
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
_ROOT = Path("/kaggle/input/datasets/lakechamplain/ripvis-cs231/ripvis_cs231")
VAL_IMAGES_DIR   = _ROOT / "val/images"
VAL_LABELS_DIR   = _ROOT / "val/labels"
TEST_IMAGES_DIR  = _ROOT / "test/images"
TEST_LABELS_DIR  = _ROOT / "test/labels"

_RUNS = Path("/kaggle/working/runs/segment")
RIPVIS_VAL_PRED_OUT  = _RUNS / "Eval/val/predictions.json"
RIPVIS_TEST_PRED_OUT = _RUNS / "Test/test/predictions.json"
RIPVIS_WEIGHTS       = _RUNS / "UIT-RipVis/YOLO26n-seg-CBAM_P4-CS231/weights/best.pt"

# ── Hyper-params ──────────────────────────────────────────────────────────────
RIPVIS_IMGSZ = 640
RIPVIS_CONF  = 0.25
IOU_TYPE     = "segm"
PR_IOU_THR   = 0.5
PR_CONF_THR  = 0.25

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR = Path("/kaggle/working/ripvis_metrics")
WRITE_INTERMEDIATE_JSON = True
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [13]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def sorted_image_paths(directory: Path) -> list[Path]:
    return sorted(p for p in directory.iterdir()
                  if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def _poly_bbox(flat: list[float]) -> list[float]:
    xs, ys = flat[0::2], flat[1::2]
    x0, x1 = min(xs), max(xs)
    y0, y1 = min(ys), max(ys)
    return [x0, y0, x1 - x0, y1 - y0]


def _poly_area(flat: list[float]) -> float:
    xs, ys = flat[0::2], flat[1::2]
    n = len(xs)
    if n < 3:
        return 0.0
    i = np.arange(n)
    j = (i + 1) % n
    return abs(np.dot(np.array(xs)[i], np.array(ys)[j]) -
               np.dot(np.array(xs)[j], np.array(ys)[i])) / 2.0


def coco_annotations_from_result(
    result: Any, image_stem: str, next_id: int
) -> tuple[list[dict[str, Any]], int]:
    anns: list[dict[str, Any]] = []
    if result is None or result.masks is None or result.boxes is None:
        return anns, next_id
    for i, xy in enumerate(result.masks.xy):
        flat = np.asarray(xy, dtype=np.float64).reshape(-1).tolist()
        if len(flat) < 6:
            continue
        cls  = int(result.boxes.cls[i].item())  if result.boxes.cls  is not None else 0
        conf = float(result.boxes.conf[i].item()) if result.boxes.conf is not None else 1.0
        anns.append({
            "id": next_id, "image_id": image_stem,
            "category_id": cls + 1, "segmentation": [flat],
            "score": conf, "bbox": _poly_bbox(flat),
            "area": _poly_area(flat), "iscrowd": 0,
        })
        next_id += 1
    return anns, next_id


if not RIPVIS_WEIGHTS.is_file():
    raise FileNotFoundError(f"Weights not found: {RIPVIS_WEIGHTS}")

yolo = YOLO(str(RIPVIS_WEIGHTS))
predict_kwargs: dict[str, Any] = dict(
    conf=RIPVIS_CONF, imgsz=RIPVIS_IMGSZ,
    verbose=False, device=[0, 1], rect=False, max_det=50,
)

tasks = (
    [(p, RIPVIS_VAL_PRED_OUT)  for p in sorted_image_paths(VAL_IMAGES_DIR)] +
    [(p, RIPVIS_TEST_PRED_OUT) for p in sorted_image_paths(TEST_IMAGES_DIR)]
)
bucket: dict[Path, list] = {RIPVIS_VAL_PRED_OUT: [], RIPVIS_TEST_PRED_OUT: []}
ann_id = 1
for img_path, out_json in tqdm(tasks, desc="RipVis inference"):
    preds = yolo.predict(source=str(img_path), **predict_kwargs)
    new_anns, ann_id = coco_annotations_from_result(
        preds[0] if preds else None, img_path.stem, ann_id
    )
    bucket[out_json].extend(new_anns)

for out_json, ann_list in bucket.items():
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps(ann_list))

for p in (RIPVIS_VAL_PRED_OUT, RIPVIS_TEST_PRED_OUT):
    if not p.is_file():
        raise FileNotFoundError(f"Missing {p}")

RipVis inference:   0%|          | 0/5586 [00:00<?, ?it/s]

In [14]:
def _parse_yolo_seg_line(line: str) -> tuple[int, list[float]] | None:
    parts = line.strip().split()
    if len(parts) < 7:
        return None
    coords = list(map(float, parts[1:]))
    return (int(float(parts[0])), coords) if len(coords) % 2 == 0 else None


def build_coco_gt_from_yolo_labels(
    images_dir: Path, labels_dir: Path, image_files: list[Path],
    *, out_json: Path | None = None,
) -> dict[str, Any]:
    images, annotations = [], []
    categories = [{"id": 1, "name": "rip", "supercategory": "rip"}]
    ann_id = 1

    for img_id, img_path in enumerate(
        tqdm(image_files, desc="Build COCO GT", leave=False), start=1
    ):
        with Image.open(img_path) as im:
            w, h = im.size
        images.append({"id": img_id, "file_name": img_path.name, "width": w, "height": h})

        lab_path = labels_dir / f"{img_path.stem}.txt"
        if not lab_path.exists():
            continue
        for raw in lab_path.read_text().strip().splitlines():
            parsed = _parse_yolo_seg_line(raw)
            if parsed is None:
                continue
            cls, coords = parsed
            pts = np.asarray(coords, dtype=np.float32).reshape(-1, 2)
            if pts.shape[0] < 3:
                continue
            pts *= [w - 1, h - 1]
            flat = pts.reshape(-1).tolist()
            annotations.append({
                "id": ann_id, "image_id": img_id, "iscrowd": 0,
                "area": _poly_area(flat), "bbox": _poly_bbox(flat),
                "category_id": cls + 1, "segmentation": [flat],
            })
            ann_id += 1

    coco = {"images": images, "annotations": annotations, "categories": categories}
    if out_json is not None:
        out_json.parent.mkdir(parents=True, exist_ok=True)
        out_json.write_text(json.dumps(coco))
    return coco

In [15]:
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
from pycocotools.cocoeval import COCOeval
from sklearn.metrics import precision_score, recall_score

_COCO_STAT_KEYS = [
    "AP[0.50:0.95]", "AP@0.50", "AP@0.75",
    "AP_small", "AP_medium", "AP_large",
    "AR@1", "AR@10", "AR@100",
    "AR_small", "AR_medium", "AR_large",
]


def _load_anns(path: str) -> list[dict[str, Any]]:
    data = json.loads(Path(path).read_text())
    anns = data if isinstance(data, list) else data["annotations"]
    for a in anns:
        a.setdefault("score", 1.0)
    return anns


def load_predictions_list(predictions_json_path: Path) -> list[dict[str, Any]]:
    data = json.loads(predictions_json_path.read_text())
    if isinstance(data, list):
        return data
    if isinstance(data, dict) and "annotations" in data:
        return data["annotations"]
    raise ValueError("Predictions must be a list or a dict with an 'annotations' key.")


def ensure_prediction_scores(anns: list[dict[str, Any]]) -> None:
    for a in anns:
        a.setdefault("score", 1.0)


def remap_prediction_image_ids(
    gt_coco_dict: dict[str, Any], preds: list[dict[str, Any]]
) -> tuple[list[dict[str, Any]], int]:
    stem_to_id = {Path(img["file_name"]).stem: int(img["id"])
                  for img in gt_coco_dict["images"]}
    out, remapped = [], 0
    for p in preds:
        q = dict(p)
        iid = q.get("image_id")
        if isinstance(iid, str):
            key = Path(iid).stem if "." in iid else iid
            if key in stem_to_id:
                q["image_id"] = stem_to_id[key]
                remapped += 1
            elif iid.isdigit():
                q["image_id"] = int(iid)
        out.append(q)
    return out, remapped


def compute_ap_map(gt_json: str, pred_json: str, iou_type: str = "segm") -> dict[str, float]:
    with redirect_stdout(io.StringIO()):
        gt_coco = COCO(gt_json)
        anns = _load_anns(pred_json)
        if not anns:
            return dict.fromkeys(_COCO_STAT_KEYS, 0.0)
        ev = COCOeval(gt_coco, gt_coco.loadRes(anns), iou_type)
        ev.evaluate(); ev.accumulate(); ev.summarize()
        stats = ev.stats
    return dict(zip(_COCO_STAT_KEYS, map(float, stats)))


def _filter_preds(anns: list[dict[str, Any]], conf_thr: float) -> list[dict[str, Any]]:
    return [dict(a, score=float(a.get("score", 1.0)))
            for a in anns if float(a.get("score", 1.0)) >= conf_thr]


def _f_beta(p: float, r: float, beta: float) -> float:
    b2 = beta * beta
    d = b2 * p + r
    return (1 + b2) * p * r / d if d > 0 else 0.0


def _pr_scores(y_true: list[int], y_pred: list[int]) -> dict[str, float]:
    if not y_true:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0, "f2": 0.0}
    p = float(precision_score(y_true, y_pred, zero_division=1))
    r = float(recall_score(y_true, y_pred, zero_division=1))
    return {"precision": p, "recall": r, "f1": _f_beta(p, r, 1), "f2": _f_beta(p, r, 2)}


def compute_pr_f1_f2(
    gt_coco: COCO, filtered_preds: list[dict[str, Any]], iou_thr: float
) -> dict[str, float]:
    if not filtered_preds:
        y_true = [1 for img_id in gt_coco.getImgIds()
                  for _ in gt_coco.loadAnns(gt_coco.getAnnIds(imgIds=img_id))]
        return _pr_scores(y_true, [0] * len(y_true))

    pred_coco = gt_coco.loadRes(filtered_preds)
    y_true, y_pred = [], []
    for img_id in gt_coco.getImgIds():
        gt_masks   = [maskUtils.decode(gt_coco.annToRLE(a))
                      for a in gt_coco.loadAnns(gt_coco.getAnnIds(imgIds=img_id))]
        pred_masks = [maskUtils.decode(pred_coco.annToRLE(a))
                      for a in pred_coco.loadAnns(pred_coco.getAnnIds(imgIds=img_id))]
        matched: set[int] = set()
        for pm in pred_masks:
            ious = [
                (np.logical_and(gm, pm).sum() / np.logical_or(gm, pm).sum()
                 if np.logical_or(gm, pm).sum() > 0 else 0.0)
                for gm in gt_masks
            ]
            best = int(np.argmax(ious)) if ious else -1
            if best >= 0 and ious[best] >= iou_thr and best not in matched:
                y_true.append(1); y_pred.append(1); matched.add(best)
            else:
                y_true.append(0); y_pred.append(1)
        y_true += [1] * (len(gt_masks) - len(matched))
        y_pred += [0] * (len(gt_masks) - len(matched))
    return _pr_scores(y_true, y_pred)


def print_ripvis_results(results: dict[str, Any]) -> None:
    print(json.dumps({"ap": results["ap"], "pr": results["pr"]}, indent=2))

In [16]:
def _write_json(path: Path, obj: Any, *, persist: bool) -> Path:
    """Write obj to path (persist=True) or a temp file (persist=False)."""
    if not persist:
        fd, tmp = tempfile.mkstemp(prefix=path.stem + "_", suffix=".json")
        os.close(fd)
        path = Path(tmp)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj))
    return path


def evaluate_split(
    split_name: str,
    images_dir: Path,
    labels_dir: Path,
    predictions_out_path: Path,
) -> dict[str, Any]:
    if not predictions_out_path.exists():
        raise FileNotFoundError(
            f"[{split_name}] Missing predictions file:\n  {predictions_out_path}"
        )
    img_files = sorted_image_paths(images_dir)
    if not img_files:
        raise RuntimeError(f"[{split_name}] No images found in {images_dir}")

    gt_dict = build_coco_gt_from_yolo_labels(images_dir, labels_dir, img_files)
    gt_path = _write_json(OUT_DIR / f"{split_name}_gt_coco.json",
                          gt_dict, persist=WRITE_INTERMEDIATE_JSON)

    preds = load_predictions_list(predictions_out_path)
    ensure_prediction_scores(preds)
    preds_mapped, n_remap = remap_prediction_image_ids(gt_dict, preds)
    mapped_path = _write_json(OUT_DIR / f"{split_name}_predictions_mapped.json",
                              preds_mapped, persist=WRITE_INTERMEDIATE_JSON)

    ap = compute_ap_map(str(gt_path), str(mapped_path), IOU_TYPE)

    with redirect_stdout(io.StringIO()):
        gt_coco = COCO(str(gt_path))

    filtered = _filter_preds(preds_mapped, PR_CONF_THR)
    pr_warning: str | None = None
    if not filtered:
        pr_warning = (
            f"[{split_name}] No predictions after score≥{PR_CONF_THR} "
            f"({len(preds)} raw → 0 kept)."
        )
        warnings.warn(pr_warning, UserWarning, stacklevel=2)

    pr = compute_pr_f1_f2(gt_coco, filtered, PR_IOU_THR)
    results: dict[str, Any] = {
        "meta": {
            "split": split_name,
            "utc_time": datetime.now(timezone.utc).isoformat(),
            "python": sys.version.split()[0],
            "platform": platform.platform(),
            "images_dir": str(images_dir), "labels_dir": str(labels_dir),
            "predictions_out_json": str(predictions_out_path),
            "gt_json_used": str(gt_path), "pred_mapped_json_used": str(mapped_path),
            "n_images_scanned": len(img_files),
            "n_gt_ann": len(gt_dict["annotations"]),
            "n_pred_raw": len(preds), "n_pred_image_id_remapped": n_remap,
            "n_pred_after_conf": len(filtered),
            "IOU_TYPE": IOU_TYPE, "PR_IOU_THR": PR_IOU_THR, "PR_CONF_THR": PR_CONF_THR,
            "WRITE_INTERMEDIATE_JSON": WRITE_INTERMEDIATE_JSON,
            "warning": pr_warning,
        },
        "ap": ap, "pr": pr,
    }
    out_path = OUT_DIR / f"results_ripvis_{split_name}.json"
    out_path.write_text(json.dumps(results, indent=2))
    return results

## Get evaluation metrics

In [17]:
# --- Validation split (RipVis AP/AR + PR/F1/F2) ---
results_val = evaluate_split(
    "val",
    VAL_IMAGES_DIR,
    VAL_LABELS_DIR,
    RIPVIS_VAL_PRED_OUT,
)
print_ripvis_results(results_val)

Build COCO GT:   0%|          | 0/2725 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
{
  "ap": {
    "AP[0.50:0.95]": 0.15789606270582107,
    "AP@0.50": 0.37269682884369326,
    "AP@0.75": 0.10510822231679905,
    "AP_small": 0.00891089108910891,
    "AP_medium": 0.17206630435764003,
    "AP_large": 0.16859290669990526,
    "AR@1": 0.16395348837209303,
    "AR@10": 0.21593992248062013,
    "AR@100": 0.21593992248062013,
    "AR_small": 0.008823529411764706,
    "AR_medium": 0.24398734177215187,
    "AR_large": 0.22339003645200486
  },
  "pr": {
    "precision": 0.5178286322511975,
    "recall": 0.47141472868217055,
    "f1": 0.4935328430129343,
    "f2": 0.480019733596448
  }
}


In [18]:
# --- Test split (RipVis AP/AR + PR/F1/F2) ---
results_test = evaluate_split(
    "test",
    TEST_IMAGES_DIR,
    TEST_LABELS_DIR,
    RIPVIS_TEST_PRED_OUT,
)
print_ripvis_results(results_test)

Build COCO GT:   0%|          | 0/2861 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
{
  "ap": {
    "AP[0.50:0.95]": 0.18017815495671333,
    "AP@0.50": 0.45741336980112857,
    "AP@0.75": 0.10077286280377966,
    "AP_small": -1.0,
    "AP_medium": 0.03166577913911923,
    "AP_large": 0.19870114313381632,
    "AR@1": 0.18107989464442492,
    "AR@10": 0.2614354697102722,
    "AR@100": 0.2614354697102722,
    "AR_small": -1.0,
    "AR_medium": 0.03460721868365181,
    "AR_large": 0.28758873929008566
  },
  "pr": {
    "precision": 0.6597088711892337,
    "recall": 0.5272168568920106,
    "f1": 0.5860680736854949,
    "f2": 0.5492796707066088
  }
}
